# train-eval-mode-branch composite — cx29: BatchNorm2d that branches on self.training (eval uses running stats)

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `inference-mode-step`, `train-eval-mode-branch`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import torch.nn as nn
import torch.nn.functional as F

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "train-eval-mode-branch"
DD_ATOM_IDS = ["inference-mode-step", "train-eval-mode-branch"]
DD_SUBTOPICS = ["PyTorch: Inference mode step", "PyTorch: train/eval mode"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## How these two atoms compose

BatchNorm has two completely different behaviours, switched by `self.training`:
- **Train mode** (`self.training is True`): normalise with the CURRENT BATCH's mean/var, and update the running stats by exponential moving average. Gradients flow through the normalisation.
- **Eval / inference mode** (`self.training is False`): normalise with the FROZEN `running_mean` / `running_var`. Do NOT update them. This is what gives BN a deterministic test-time function, independent of batch composition.

**The two atoms.**
- **train-eval-mode-branch** — the `if self.training: ... else: ...` switch inside `forward`. Without this branch, a batch of 1 image at test time gets `var = 0` and the output is NaN.
- **inference-mode-step** — the eval branch. It wraps the forward in `t.no_grad()` or uses `t.inference_mode()` to avoid building autograd machinery the inference path doesn't need. (`.eval()` flips `self.training`; `t.inference_mode()` is the autograd-side switch.)

**Anatomy.**
```python
def forward(self, x):
    if self.training:
        mu = x.mean(dim=(0, 2, 3)); var = x.var(dim=(0, 2, 3), unbiased=False)
        # ... update running stats ...
    else:
        # inference-mode-step: use frozen buffers, no autograd.
        mu = self.running_mean; var = self.running_var
    x_hat = (x - mu[None,:,None,None]) / t.sqrt(var[None,:,None,None] + eps)
    return gamma * x_hat + beta
```

**Why care.** Forgetting `model.eval()` before validation is one of the most common ResNet bugs. Test loss looks worse than train loss because BN is still using each validation batch's stats (often differently distributed).

### Composite Exercise — BatchNorm2d that branches on self.training (eval uses running stats)

**Atoms exercised together**: `inference-mode-step`, `train-eval-mode-branch`

Implement `cx29_make_bn_with_branch()` — return `MyBN(nn.Module)` with the train/eval branch wired in.

Required structure (extends cx28):
- `__init__(self, num_features, eps=1e-5, momentum=0.1)` — same as cx28: `weight`, `bias` as Parameters; `running_mean`, `running_var` as buffers.
- `forward(self, x)` — **branch on `self.training`**:
  - If training: compute batch stats, UPDATE running stats, normalise with BATCH stats.
  - If eval: normalise with `self.running_mean` / `self.running_var`, do NOT update.
  - In BOTH branches: apply affine (`gamma * x_hat + beta`).

The test:
- Sets `bn.training = True`, runs forward, records `running_mean` / `running_var` changes.
- Sets `bn.training = False`, runs forward AGAIN — running stats MUST NOT change, and the output MUST equal `F.batch_norm(..., training=False, ...)`.
- Sanity: train-mode output for a uniform-mean batch != eval-mode output (different mu/var).
- The eval branch is also tested under `t.no_grad()` to confirm it works in inference mode.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx29_make_bn_with_branch():
    """Return the MyBN class with train/eval branch."""
    raise NotImplementedError

def _test_cx29():
    MyBN = cx29_make_bn_with_branch()
    assert issubclass(MyBN, nn.Module)

    t.manual_seed(0)
    bn = MyBN(num_features=3, eps=1e-5, momentum=0.1)
    # Push running stats away from defaults so we can tell train- vs eval-mode apart.
    with t.no_grad():
        bn.running_mean.copy_(t.tensor([0.5, -0.5, 1.0]))
        bn.running_var.copy_(t.tensor([2.0, 0.5, 1.5]))
        bn.weight.copy_(t.tensor([1.2, 0.8, 1.5]))
        bn.bias.copy_(t.tensor([-0.1, 0.2, 0.3]))

    x = t.randn(4, 3, 5, 6) + 3.0  # nonzero mean — so batch vs running stats differ.

    # Case A: TRAIN mode — output matches F.batch_norm(training=True).
    bn.train()  # sets self.training = True.
    assert bn.training is True
    rm_before = bn.running_mean.clone()
    rv_before = bn.running_var.clone()
    rm_ref = rm_before.clone()
    rv_ref = rv_before.clone()
    ref_train = F.batch_norm(x, rm_ref, rv_ref, bn.weight, bn.bias, training=True, momentum=0.1, eps=1e-5)
    out_train = bn(x)
    assert t.allclose(out_train, ref_train, atol=1e-5), 'train-mode output != F.batch_norm reference'
    # Running stats moved.
    assert not t.allclose(bn.running_mean, rm_before), 'train mode must update running_mean'

    # Case B: EVAL mode — output uses running stats; stats DO NOT update.
    bn.eval()  # sets self.training = False.
    assert bn.training is False
    rm_before_eval = bn.running_mean.clone()
    rv_before_eval = bn.running_var.clone()
    ref_eval = F.batch_norm(
        x, bn.running_mean, bn.running_var, bn.weight, bn.bias,
        training=False, eps=1e-5,
    )
    out_eval = bn(x)
    assert t.allclose(out_eval, ref_eval, atol=1e-5), 'eval-mode output != F.batch_norm(training=False) reference'
    assert t.allclose(bn.running_mean, rm_before_eval), 'eval mode must NOT update running_mean'
    assert t.allclose(bn.running_var, rv_before_eval), 'eval mode must NOT update running_var'

    # Case C: train and eval outputs differ (proves the branch matters).
    assert not t.allclose(out_train, out_eval), (
        'train- and eval-mode outputs are identical — the branch on self.training is missing'
    )

    # Case D: inference-mode-step — eval path must work under t.no_grad().
    bn.eval()
    rm_before_ng = bn.running_mean.clone()
    with t.no_grad():
        out_ng = bn(x)
    # Same output as Case B.
    assert t.allclose(out_ng, ref_eval, atol=1e-5)
    # Output should have no grad_fn (we were under no_grad).
    assert out_ng.requires_grad is False, 'eval forward under t.no_grad() should not require grad'
    # And running stats still untouched.
    assert t.allclose(bn.running_mean, rm_before_ng), 'inference-mode forward must not touch running stats'
    _dd_passed.add('cx29')

_test_cx29()

<details><summary>Show solution — cx29</summary>

```python
def cx29_make_bn_with_branch():
    class MyBN(nn.Module):
        def __init__(self, num_features, eps=1e-5, momentum=0.1):
            super().__init__()
            self.eps = eps
            self.momentum = momentum
            self.weight = nn.Parameter(t.ones(num_features))
            self.bias = nn.Parameter(t.zeros(num_features))
            self.register_buffer('running_mean', t.zeros(num_features))
            self.register_buffer('running_var', t.ones(num_features))

        def forward(self, x):
            # Atom A (train-eval-mode-branch): self.training switches the stat source.
            if self.training:
                mu = x.mean(dim=(0, 2, 3))
                var_biased = x.var(dim=(0, 2, 3), unbiased=False)
                # Update running stats with the unbiased var (PyTorch convention).
                n = x.shape[0] * x.shape[2] * x.shape[3]
                var_unbiased = var_biased * (n / (n - 1)) if n > 1 else var_biased
                self.running_mean.mul_(1 - self.momentum).add_(mu.detach() * self.momentum)
                self.running_var.mul_(1 - self.momentum).add_(var_unbiased.detach() * self.momentum)
                mu_used, var_used = mu, var_biased
            else:
                # Atom B (inference-mode-step): frozen running stats; no update; works under no_grad.
                mu_used, var_used = self.running_mean, self.running_var
            x_hat = (x - mu_used[None, :, None, None]) / t.sqrt(var_used[None, :, None, None] + self.eps)
            return self.weight[None, :, None, None] * x_hat + self.bias[None, :, None, None]

    return MyBN
```

Two flags often get confused: `self.training` (Module-level, flipped by `.train()/.eval()`) and `t.is_grad_enabled()` (global, flipped by `t.no_grad()` / `t.inference_mode()`). The BN branch above only looks at `self.training` — autograd-disabled inference is orthogonal and works for free because we never touch a Parameter that needs grad on the eval path. The `.detach()` on `mu` and `var_unbiased` when updating buffers prevents the running-stat update from accidentally building a gradient back through the buffer.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx29'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx29',
        'subtopics': ["PyTorch: Inference mode step", "PyTorch: train/eval mode"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()